## Yapay Sinir Ağları (Model)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, log_loss
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
diabetes = pd.read_csv('diabetes.csv')
df = diabetes.copy()
df.dropna()
y = df['Outcome']
X = df.drop(['Outcome'], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size = 0.30,
                                                    random_state = 42)

In [3]:
#standartlastirma

In [4]:
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

In [5]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
mlpc = MLPClassifier().fit(X_train_scaled, y_train)

## Tahmin

In [7]:
#test hatasi

In [8]:
y_pred = mlpc.predict(X_test_scaled)
accuracy_score(y_test, y_pred)

0.7316017316017316

## Model Tuning

In [9]:
mlpc.get_params()

{'activation': 'relu',
 'alpha': 0.0001,
 'batch_size': 'auto',
 'beta_1': 0.9,
 'beta_2': 0.999,
 'early_stopping': False,
 'epsilon': 1e-08,
 'hidden_layer_sizes': (100,),
 'learning_rate': 'constant',
 'learning_rate_init': 0.001,
 'max_fun': 15000,
 'max_iter': 200,
 'momentum': 0.9,
 'n_iter_no_change': 10,
 'nesterovs_momentum': True,
 'power_t': 0.5,
 'random_state': None,
 'shuffle': True,
 'solver': 'adam',
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': False,
 'warm_start': False}

In [10]:
mlpc_params = {"alpha" : [0.1, 0.01, 0.02, 0.005, 0.0001, 0.00001],
               "hidden_layer_sizes" : [(10,10,10),
                                      (100,100,100),
                                      (100,100),
                                      (3,5),
                                      (5,3)],
               "solver" : ["lbfgs", "adam", "sgd"],
               "activation" : ["relu", "logistic"]}

In [11]:
mlpc = MLPClassifier()

mlpc_cv_model = GridSearchCV(mlpc, mlpc_params,
                             cv = 10,
                             n_jobs = -1,
                             verbose = 2)

mlpc_cv_model.fit(X_train_scaled, y_train)

Fitting 10 folds for each of 180 candidates, totalling 1800 fits


GridSearchCV(cv=10, estimator=MLPClassifier(), n_jobs=-1,
             param_grid={'activation': ['relu', 'logistic'],
                         'alpha': [0.1, 0.01, 0.02, 0.005, 0.0001, 1e-05],
                         'hidden_layer_sizes': [(10, 10, 10), (100, 100, 100),
                                                (100, 100), (3, 5), (5, 3)],
                         'solver': ['lbfgs', 'adam', 'sgd']},
             verbose=2)

In [12]:
print("En iyi parametre değerleri : " + str(mlpc_cv_model.best_params_))

En iyi parametre değerleri : {'activation': 'relu', 'alpha': 0.1, 'hidden_layer_sizes': (100, 100), 'solver': 'sgd'}


In [13]:
#final model

In [14]:
mlpc_tuned = MLPClassifier(activation = "relu",
                           alpha = 0.1,
                           hidden_layer_sizes = (100,100,100),
                           solver = "sgd").fit(X_train_scaled, y_train)

In [15]:
#final test hatasi

In [16]:
y_pred = mlpc_tuned.predict(X_test_scaled)

In [17]:
accuracy_score(y_test, y_pred)

0.7619047619047619